# 07 — Final Strategy Report

Comprehensive performance report: all metrics, alpha/beta decomposition, factor exposure, drawdown analysis, and plots.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


In [ ]:
from statarb.signals.momentum import MomentumSignals
from statarb.signals.reversal import ReversalSignals
from statarb.signals.activity import ActivityFilter
from statarb.backtest.execution import ExecutionModel
from statarb.backtest.weighting import StrategyWeighting
from statarb.evaluation.metrics import PerformanceMetrics
from statarb.evaluation.risk import RiskAnalytics
from statarb.evaluation.plots import PerformancePlots
from statarb.evaluation.reporting import PerformanceReporter
from experiments._utils import run_bt

mom = MomentumSignals()
rev = ReversalSignals()
act = ActivityFilter()
em  = ExecutionModel(market_order_cost=0.0020)
pm  = PerformanceMetrics()
ra  = RiskAnalytics()
pp  = PerformancePlots()
sw  = StrategyWeighting()


## Build Composite Strategy

In [ ]:
# Build and backtest component signals
components = {
    "mom_6_1":     fe.cross_sectional_rank(act.activity_gate(mom.momentum_6_1(returns), volume)),
    "mom_3_1":     fe.cross_sectional_rank(act.activity_gate(mom.momentum_3_1(returns), volume)),
    "sharpe_mom":  fe.cross_sectional_rank(act.activity_gate(mom.sharpe_momentum(returns,63), volume)),
    "reversal_5d": fe.cross_sectional_rank(act.activity_gate(rev.weekly_reversal(returns), volume)),
    "vol_adj_rev": fe.cross_sectional_rank(act.activity_gate(rev.vol_adjusted_reversal(returns), volume)),
}

comp_returns = {}
comp_turnover = {}
for name, sig in components.items():
    net_rets, result = run_bt(sig, returns, em)
    comp_returns[name] = net_rets
    comp_turnover[name] = result["turnover"]

comp_df = pd.DataFrame(comp_returns).dropna()
composite = sw.equal_weight(comp_df)
print(f"Composite strategy: {len(composite)} days of returns")


## Full Performance Report

In [ ]:
# BTC or first available asset as benchmark
if "BTC/USDT" in prices.columns:
    benchmark = fe.log_returns(prices[["BTC/USDT"]])["BTC/USDT"]
elif len(prices.columns) > 0:
    benchmark = fe.log_returns(prices[[prices.columns[0]]])[prices.columns[0]]
else:
    benchmark = None

report = pm.full_report(composite, benchmark_returns=benchmark)
print(report.to_string())


## Alpha / Beta Analysis

In [ ]:
if benchmark is not None:
    ab = ra.alpha_beta(composite, benchmark)
    print(f"Alpha (annualized): {ab['alpha']*100:.2f}%")
    print(f"Beta:               {ab['beta']:.3f}")
    print(f"R²:                 {ab['r_squared']:.3f}")
    print(f"Alpha t-stat:       {ab['alpha_tstat']:.2f}")
    print()

    # Rolling beta plot
    rolling_beta = ra.rolling_beta(composite, benchmark, window=63)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(rolling_beta.index, rolling_beta.values, linewidth=1.2, color="steelblue")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axhline(ab["beta"], color="red", linewidth=0.8, linestyle=":",
               label=f"Full-sample β={ab['beta']:.2f}")
    ax.set_title("Rolling Beta vs Benchmark (63-day)", fontweight="bold")
    ax.set_ylabel("Beta")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()


## Drawdown Analysis

In [ ]:
dd_table = ra.drawdown_analysis(composite)
if not dd_table.empty:
    print(f"Drawdown episodes exceeding 5%: {len(dd_table)}")
    print(dd_table.to_string())
else:
    print("No drawdown episodes exceeding 5% (synthetic data may show no deep drawdowns)")

# Drawdown plot
fig = pp.drawdown_plot(composite)
plt.show()


## All Performance Plots

In [ ]:
fig = pp.cumulative_return_plot(composite, benchmark=benchmark,
                                 title="Composite Strategy Cumulative Return")
plt.show()

fig = pp.return_distribution(composite)
plt.show()

fig = pp.rolling_sharpe_plot(composite, window=63)
plt.show()

try:
    fig = pp.monthly_returns_heatmap(composite)
    plt.show()
except Exception as e:
    print(f"Monthly heatmap requires multi-month data: {e}")


## Component Attribution

In [ ]:
# Side-by-side comparison of all components + composite
all_strats = dict(comp_returns)
all_strats["composite"] = composite

reporter = PerformanceReporter(
    all_strats,
    benchmark_returns=benchmark,
    periods_per_year=365,
)
reporter.print_report(title="Final Strategy Evaluation")


## Summary

| Metric | Target | Achieved |
|--------|--------|----------|
| Sharpe Ratio (net) | 1.0–2.0 | see above |
| Annual Return (net) | 15–25% | see above |
| Max Drawdown | < 20% | see above |
| Beta to BTC | < 0.3 | see above |

See `docs/strategy_summary.md` for full write-up.